# 2 · Training: learn from past outcomes

| [1. Ingest](1_ingest.ipynb) | **2. Training** (this notebook) | [3. Inference](3_inference.ipynb) |
| --- | --- | --- |
| Load the tables and build the index | Learn from past outcomes and save the model | Score every customer and build the call list |

**The business question** for all three notebooks: *which customers will place an order in the next 30 days?* The answer is a ranked call list the account team can work from.

**This notebook in one sentence:** show the model thousands of past situations where we already know what happened, so it learns which patterns come before an order, then save it.

| | |
| --- | --- |
| **Goes in** | The index from notebook 1, and the business question written as one line |
| **Comes out** | A trained model saved in `models/orders-next-30d/` (kept out of git; step 5 shows its size), and its exam results |
| **In RAG terms** | RAG has no such step, because its LLM arrives already trained. No ready-made model can read *your* order history, so we train a small one from scratch, like fine-tuning that starts from zero |
| **Run time** | About 15 seconds on a laptop CPU |

In [6]:
import shop_demo as shop   # first: it also hides known, harmless library warnings
import kumo.engine as kumo
from kumo_bundle import save_bundle, describe   # save and reload helpers, see kumo_bundle.py

shop.start()
graph = shop.build_graph()
snapshot = graph.snapshot()   # reopens notebook 1's index; nothing is rebuilt

## Step 1 · Ask the business question

Kumo's predictive query language (PQL) states the question in one line: for each customer, will the count of orders in the next 30 days be above zero?
Kumo checks the line against the tables and recognises it as a yes/no question.

In [7]:
question = kumo.PredictiveQuery(graph, shop.QUESTION)
question.validate()
print(shop.QUESTION)
print("Kind of question:", question.get_task_type())

PREDICT COUNT(orders.*, 0, 30, days) > 0 FOR EACH customers.customer_id
Kind of question: binary_classification


## Step 2 · Turn history into practice questions

Kumo steps back through the order history 30 days at a time.
At each date it writes one practice question per customer, "as of this date, will they order in the next 30 days?", together with the answer that actually happened.

The important rule: while the model studies a question, it sees only the orders up to that date, never the answer period.
In RAG terms, every practice question gets its own retrieval, limited to what was known on that date.

In [8]:
practice = question.generate_training_table(snapshot)
practice_rows = shop.practice_summary(practice)

10,500 practice questions = 500 customers × 21 dates, every 30 days from 2025-02-23 to 2026-10-16
Answered 'yes, ordered within 30 days': 7.8%


For example, Acme's last three practice questions.
The date and the answer are Kumo's; the two order columns are read from the order file to show both sides.

In [9]:
shop.flashcards(practice_rows, "C001")

,Orders the model may look at,Answer (Kumo): ordered in the next 30 days?,The orders behind the answer
As of,,,
2026-08-17,none yet,no,none
2026-09-16,none yet,yes,"[988, 1017]"
2026-10-16,"[988, 1017]",yes,[1043]


On 16 September Acme had no orders yet, but two arrived in the next 30 days, so the answer is yes.
On 16 October those two orders are history the model may look at, and order 1043 (2 November) is the answer.

## Step 3 · Train

The settings, in plain words:

- **What it reads:** each customer's 10 most recent orders up to the question's date, and the products in them, plus the customer's region and signup date.
- **The model:** Kumo's built-in graph neural network. It reads a customer's records by following the links, from products to orders to the customer.
- **Two candidates:** Kumo's FAST preset trains 2 versions with different settings and keeps the one that scores better on held-back questions.
- **Fixed random seed:** keeps reruns similar but not identical. A customer's score can move by several points between runs, enough to swap accounts with close scores. That's one reason to save the model in step 5 and score from it, rather than retrain for every list.
- **An exam it never studies:** the questions from the newest 2 dates are held back for step 4.

In [10]:
trainer, result = shop.train(question, snapshot, practice)

Trained in 12 seconds on the laptop CPU


## Step 4 · How good is it?

The exam is the newest practice questions, which the model never saw while training.
The three rows answer three business questions: does it put buyers above non-buyers, is the top of its list worth calling first, and can its probabilities be added up into a forecast?
**Synthetic data:** these numbers show the method works on a pattern built into the generator, not how a model would do on real accounts.

In [11]:
shop.exam_results(result)

Exam: 1,000 practice questions from the newest dates (2026-09-16, 2026-10-16), never seen in training; 7.5% answered yes


,Result,In plain words
Measure,,
Ranking (AUROC),0.81,"given one customer who ordered and one who didn't, the model ranks the buyer higher 81% of the time (a coin flip would be 50%)"
Top of the list,38% vs 7.5%,customers in the model's top 10% ordered at 5.1× the average rate
Forecast,94 predicted vs 75 actual,"over the exam dates, the model's probabilities added up to 94 customers ordering, and 75 did: +25%, so treat its forecast as a rough guide"


## Step 5 · Save the model

The trained model is a folder: its learned numbers (the weights) plus the instructions for using them, such as how each column is turned into numbers, the question it answers and the 50% yes/no threshold.
Notebook 3 loads it from here without retraining.

The folder stays out of git: it is rebuilt from the data in seconds, it changes on every training run, and it is derived from customer data.
For a real model, keep each version in object storage or a model registry instead.

In [12]:
save_bundle(result, shop.MODEL_DIR, threshold=0.5, overwrite=True)   # overwrite: this demo retrains on every run
print(describe(shop.MODEL_DIR))

Model bundle: models/orders-next-30d
Question (PQL): PREDICT COUNT(orders.*, 0, 30, days) > 0 FOR EACH customers.customer_id
Trained: 2026-11-24 07:46 UTC with kumo-engine 1.0.0.dev0+5d17fb3 (torch 2.15.0, Python 3.14.6)
Task type: binary_classification
Scoring defaults: binary_classification_threshold 0.5
Test results: ap 0.372, auprc 0.368, auroc 0.808
Validation results: ap 0.299, auprc 0.291, auroc 0.788
Trained on:
  customers: key=customer_id time=None columns=['region', 'signup_date']
  products: key=product_id time=None columns=['name', 'category', 'price']
  orders: key=order_id time=order_date columns=['customer_id', 'product_id', 'amount'] links=customer_id->customers, product_id->products
Files: 17, 17.7 MB (weights: 17.6 MB)


## Next

[Notebook 3](3_inference.ipynb) loads this saved model, scores every customer as of 15 November 2026 and builds the call list.